In [ ]:
import numpy as np
import torch
import torch.nn as nn

from dlfs_autograd.layers import ConvLayer, ConvTransposeLayer
from dlfs_autograd.tensor.base import Parameter
from .helpers import compare_arrays

In [ ]:
use_autograd = True

np.random.seed(0)
torch.manual_seed(0)

B = 2
C_in = 5
H_in = 18
W_in = 23

C_out = 9

kernel_size=8
stride=3
padding=3

x = np.random.normal(size=(B, C_in, H_in, W_in))
print(x.shape)

In [ ]:
conv = nn.Conv2d(C_in, C_out, kernel_size=kernel_size, stride=stride, padding=padding)
convt = nn.ConvTranspose2d(C_out, C_in, kernel_size=kernel_size, stride=stride, padding=padding)

with torch.no_grad():
    conv.bias.zero_()
    convt.bias.zero_()

torch_x = torch.Tensor(x)

output_conv = conv(torch_x)
print(f'Conv out: {output_conv.shape}')

output_convt = convt(output_conv)
print(f'ConvT out: {output_convt.shape}')

In [ ]:
kernel1 = conv.weight.detach().numpy()
kernel2 = convt.weight.detach().numpy()

print(f'kernel shapes: {kernel1.shape}, {kernel2.shape}')

if use_autograd:
    kernel1 = kernel1.reshape(C_out, C_in*kernel_size*kernel_size)
    kernel2 = kernel2.transpose(1, 2, 3, 0).reshape(C_in*kernel_size*kernel_size, C_out)
    print(f'kernel shapes: {kernel1.shape}, {kernel2.shape}')

In [ ]:
my_conv = ConvLayer(C_in, C_out, kernel_size=kernel_size, stride=stride, padding=padding)
my_convt = ConvTransposeLayer(C_out, C_in, kernel_size=kernel_size, stride=stride, padding=padding)
print(f'Init conva kernel: {my_conv.kernels.shape}')
print(f'Init convta kernel: {my_convt.kernels.shape}')

my_conv.kernels = kernel1
my_convt.kernels = kernel2

if use_autograd:
    my_conv.kernels = Parameter(
        kernel1
    )

    my_convt.kernels = Parameter(
        kernel2
    )

print(f'Nakon conva kernel: {my_conv.kernels.shape}')
print(f'Nakon convta kernel: {my_convt.kernels.shape}')

In [ ]:
my_conv.forward(x)
print(f'Conv out: {my_conv.output.shape}')

my_convt.forward(my_conv.output)
print(f'Conv out: {my_convt.output.shape}')


In [ ]:
conv_out = my_conv.output
convt_out = my_convt.output

if use_autograd:
    conv_out = my_conv.output.data
    convt_out = my_convt.output.data

print(np.allclose(output_conv.detach().numpy(), conv_out, atol=1e-6, rtol=1e-6))
print(np.max(np.abs(output_conv.detach().numpy() - conv_out)))

In [ ]:
print(np.allclose(output_convt.detach().numpy(), convt_out, atol=1e-6, rtol=1e-6))
print(np.max(np.abs(output_convt.detach().numpy() - convt_out)))

In [ ]:
#res_conv_torch = torch.sum(output_conv)
#print(res_conv_torch)
#res_conv_torch.backward()
#print(conv.weight.grad.shape)
#print(conv.bias.grad.shape)

In [ ]:
#res_convmy = my_conv.output.sum()
#print(res_convmy)
#res_convmy.backward()
#print(my_conv.kernels.grad.shape)

In [ ]:
#print(np.allclose(conv.weight.grad.detach().numpy().reshape(C_out, -1), my_conv.kernels.grad, atol=1e-5, rtol=1e-5))
#print(np.max(np.abs(conv.weight.grad.detach().numpy().reshape(C_out, -1) - my_conv.kernels.grad)))

In [ ]:
res_convt_torch = torch.sum(output_convt)
print(res_convt_torch)
res_convt_torch.backward()
print(convt.weight.grad.shape)

In [ ]:
res_convt_my = my_convt.output.sum()
print(res_convt_my)
res_convt_my.backward()
print(my_convt.kernels.grad.shape)

In [ ]:
torch_grad = (
    convt.weight.grad.detach().cpu().numpy()
    .transpose(1, 2, 3, 0)
    .reshape(-1, C_out)
)

print(np.allclose(torch_grad, my_convt.kernels.grad, atol=1e-5, rtol=1e-5))
print(np.max(np.abs(torch_grad - my_convt.kernels.grad)))